# New Working

In [ ]:
# src/FRAME_FM/dataloaders/era5_kerchunk_dataloader.py

from __future__ import annotations

from typing import Callable, Optional, Sequence

import torch
from torch.utils.data import TensorDataset
import xarray as xr
import numpy as np

from FRAME_FM.utils.LightningDataModuleWrapper import BaseDataModule
from FRAME_FM.datasets.InputOnly_Dataset import TransformedInputDataset, TransformedInputCoordsDataset


#3 classes for 3 different Input Positioning options in mmmae - ERA5TiledBaseDataModule: no position info, just values (for inputs_positioned=None), 
# ERA5SpatialPixelsDataModule: per-pixel lat/lon grids (for inputs_positioned="pixels")
# ERA5SpatialBoundsDataModule: tile bounds (for inputs_positioned="bounds")


class ERA5TiledBaseDataModule(BaseDataModule):
    """
    A simple DataModule for ERA5 data opened via Kerchunk. 
    Opens the ERA5 dataset using the kerchunk reference JSON. 
    Chooses a set of variables (e.g., t2m, u10, v10, msl...).
    For each timestep, it cuts the global lat/lon grid into small square "tiles".
    Each tile becomes one training example for an autoencoder.

    The model then learns to compress (encode) and reconstruct (decode) these tiles.

    Notes:
    - Output tiles are shaped like (C, H, W) where:
        C = number of variables (channels)
        H = tile_size_lat
        W = tile_size_lon
    - If you use the positional embedders in mmmae (inputs_positioned="pixels"/"bounds"),
      use one of the subclasses below that also returns coordinates.
    """

    train_dataset: torch.utils.data.Dataset
    val_dataset: torch.utils.data.Dataset
    test_dataset: torch.utils.data.Dataset | None

    def __init__(
        self,
        data_root: str,
        # what to load
        variables: Sequence[str],
        time_min: Optional[str] = None,  # e.g. "2000-01-01"
        time_max: Optional[str] = None,  # e.g. "2000-12-31"
        # tiling
        tile_size_lat: int = 64,
        tile_size_lon: int = 64,
        # optional longitude handling
        convert_longitude_to_180: bool = True,
        # standard datamodule args
        batch_size: int = 32,
        num_workers: int = 4,
        pin_memory: bool = True,
        persistent_workers: bool = False,
        train_split: float = 0.85,
        val_split: float = 0.15,
        test_split: float = 0.0,
        split_strategy: str = "fraction",
        train_transforms: Callable | None = None,
        val_transforms: Callable | None = None,
        test_transforms: Callable | None = None,
        # optional explicit indices
        train_indices: Optional[Sequence[int]] = None,
        val_indices: Optional[Sequence[int]] = None,
        test_indices: Optional[Sequence[int]] = None,
        # optional xarray chunking (helps performance)
        chunks: Optional[dict] = None,
    ) -> None:
        super().__init__(
            data_root=data_root,
            batch_size=batch_size,
            num_workers=num_workers,
            pin_memory=pin_memory,
            persistent_workers=persistent_workers,
            train_split=train_split,
            val_split=val_split,
            test_split=test_split,
            split_strategy=split_strategy,
            train_transforms=train_transforms,
            val_transforms=val_transforms,
            test_transforms=test_transforms,
            train_indices=train_indices,
            val_indices=val_indices,
            test_indices=test_indices,
        )

        self.variables = list(variables)
        self.time_min = time_min
        self.time_max = time_max
        self.tile_size_lat = tile_size_lat
        self.tile_size_lon = tile_size_lon
        self.convert_longitude_to_180 = convert_longitude_to_180
        self.chunks = chunks

    def _load_raw_data(self) -> xr.DataArray:
        """
        Open ERA5 via the kerchunk reference (data_root is the kerchunk JSON URL/path),
        select variables and time window, and return a single DataArray with a 'channel' dim.

        Returned DataArray dims (typical):
            channel, time, latitude, longitude
        """
        # ds = xr.open_dataset(self.data_root, engine="kerchunk", chunks=self.chunks)
        ds = xr.open_dataset(self.data_root, engine="kerchunk")

        # Select variables (channels)
        missing = [v for v in self.variables if v not in ds.data_vars]
        if missing:
            raise ValueError(
                f"Some requested variables are not present in dataset: {missing}. "
                f"Available variables include: {list(ds.data_vars)[:20]}..."
            )
        ds = ds[self.variables]

        # Select time range (optional)
        if self.time_min is not None or self.time_max is not None:
            t0 = self.time_min if self.time_min is not None else ds["time"].min().item()
            t1 = self.time_max if self.time_max is not None else ds["time"].max().item()
            ds = ds.sel(time=slice(t0, t1))

        # Optionally convert longitude from [0, 360) to [-180, 180)
        # This can be useful so "position_space" is more intuitive.
        if self.convert_longitude_to_180 and "longitude" in ds.coords:
            lon = ds["longitude"]
            lon_180 = ((lon + 180) % 360) - 180
            ds = ds.assign_coords(longitude=lon_180).sortby("longitude")

        # Convert multi-variable Dataset to a single DataArray with 'channel'
        #Two separate 3D arrays: d2m(time, lat, lon) & t2m(time, lat, lon) --> one 4D array arr(channel = nvars = 2(here), time, lat, lon) where arr[0] = d2m and arr[1] = t2m.
        #ML models expect (batch, channel, height, width) or (channel,time,lat,lon) rather than d2m etc. 
        # print(ds)
        arr = ds.to_array(dim="channel")
        
        #Store global lat/lon for later use in positional encoding (e.g., to compute real-world coordinates from tile-local coords)
        self._global_lat = arr["latitude"].values
        self._global_lon = arr["longitude"].values
        # print(self._global_lat)
        # print(arr)

        # Ensure we have the expected coordinate names(ERA5 is typically latitude/longitude, this keeps it more robust.)
        for req in ["time", "latitude", "longitude"]:
            if req not in arr.coords and req not in arr.dims:
                raise ValueError(
                    f"Expected ERA5-style coord/dim '{req}' not found. "
                    f"Found dims={arr.dims}, coords={list(arr.coords)}"
                )

        # Make sure channel is first for easier downstream use
        # (channel, time, latitude, longitude)
        arr = arr.transpose("channel", "time", "latitude", "longitude")
        # print(arr)

        return arr

    def _tile_array(self, arr: xr.DataArray) -> xr.DataArray:
        """
        Turn (channel, time, latitude, longitude) into a batch of tiles:
            (batch_dim, channel, tile_lat, tile_lon)

        How tiling works (simple explanation):
        - Imagine the global map is a big rectangle of pixels.
        - We chop it into small squares (tiles).
        - We do this for every timestep, so the final dataset is:
            "all tiles from all times".
        """
        # basic sanity checks
        if "latitude" not in arr.dims or "longitude" not in arr.dims:
            raise ValueError(f"Expected latitude/longitude dims, got {arr.dims}.")

        n_lat = arr.sizes["latitude"]
        n_lon = arr.sizes["longitude"]
        if n_lat < self.tile_size_lat or n_lon < self.tile_size_lon:
            raise ValueError(
                "ERA5 grid is smaller than the requested tile size: "
                f"{n_lat}x{n_lon} < {self.tile_size_lat}x{self.tile_size_lon}"
            )

        # print(f"Tiling array of shape {arr.shape} into tiles of size "
        #       f"{self.tile_size_lat}x{self.tile_size_lon}...")
        # print(n_lat)
        # print(n_lon)
        
        #Create tiles --> (latitude × longitude)  e.g. 721 × 1440 --> (tile_lat × tile_lon) e.g. 64 × 64, so we get small tiles per time step.
        # This creates a structured view of the global grid as (tile IDs + local pixel offsets) and then extract_pixel_positions converts that structured view into real-world coordinates. 
        # It will bascially tell model -  For this variable and time, look at the tile located at (tile_lat_id, tile_lon_id) in the global tile grid, and then take pixel (tile_lat, tile_lon) inside that tile.
        #Before coarsen, arr dims: (channel, time, latitude, longitude)
        # print("Before coarsen \n", arr)
        
        #Creates a window of defined tile size to slide across the latitude and longitude dimensions, effectively grouping the data into tiles. The 'boundary="pad"' argument ensures that if the total size of the latitude or longitude dimension is not perfectly divisible by the tile size, the remaining pixels will be padded (usually with NaNs) to create a full tile.
        tiles = arr.coarsen(latitude=self.tile_size_lat, longitude=self.tile_size_lon, boundary="pad")
        #After coarsen, arr = [windows->{'latitude': tile_size, 'longitude': tile_size},side->left] 
        # print("After coarsen \n", tiles)
        
        #Takes each grouped axis and splits it into two dimensions ==> latitude = num_lat turned to tile_lat_id (given by number_of_tiles_in_lat ex. 721/64 ~=12, tile_lat_id = 0 → rows 0–63)  and tile_lat = which pixel inside that tile? (tile_lat = 0 → first row inside tile (local pixel coordinate inside the tile)); same for lon
        tiles = tiles.construct(latitude=("tile_lat_id", "tile_lat"), longitude=("tile_lon_id", "tile_lon"))
        #After construct tile dims: ('channel', 'time', 'tile_lat_id', 'tile_lat', 'tile_lon_id', 'tile_lon')
        # print("After construct tile dims:", tiles.dims, tiles.shape)
        
        # Unflatten using stack => time * tile_lat_id * tile_lon_id becomes batch_dim, so we get a big batch of tiles across all time steps.
        tiles = tiles.stack(batch_dim=("time", "tile_lat_id", "tile_lon_id"))
        # After stack tile dims: ('channel', 'tile_lat', 'tile_lon', 'time * tile_lat_id * tile_lon_id = batch_dim') 
        # print("After stack tile dims:", tiles.dims, tiles.shape)
        
        # Reorder to what PyTorch typically expects
        tiles = tiles.transpose("batch_dim", "channel", "tile_lat", "tile_lon")
        #After transpose tile dims: ('batch_dim', 'channel', 'tile_lat', 'tile_lon')
        # print("After transpose tile dims:", tiles.dims)
        
        # print(f"Resulting tiled array shape: {tiles.shape} {tiles.dims}")
        #Resulting tiled array shape: (N, C, tile_size_lat, tile_size_lon) where N = number of tiles across all time steps, C = number of variables (channels)
        
        # Replace padded NaNs with zeros (often required for stable training / masking)
        tiles = tiles.fillna(0)

        return tiles
    
    #Function to check - Our tiles come from stacking (time, tile_lat_id, tile_lon_id) into batch_dim. 
    # That means we can pick a tile and reconstruct the exact global slice indices and compare to the original arr. 
    def assert_tile_matches_global(self, tile_index: int = 0, channel: int = 0, atol: float = 1e-5):
        """
        Proves that tiles[tile_index] == arr[channel, time, lat_slice, lon_slice]
        for the corresponding (time, tile_lat_id, tile_lon_id) tuple.
        """

        arr = self._raw_data  # (channel,time,latitude,longitude)
        tiles = self._tile_array(arr)  # (batch_dim, channel, tile_lat, tile_lon)

        bt = tiles["batch_dim"].values[tile_index]  # (time, tile_lat_id, tile_lon_id)
        t_val, tile_lat_id, tile_lon_id = bt
        tile_lat_id = int(tile_lat_id)
        tile_lon_id = int(tile_lon_id)

        H = tiles.sizes["tile_lat"]
        W = tiles.sizes["tile_lon"]
        lat_start = tile_lat_id * self.tile_size_lat
        lon_start = tile_lon_id * self.tile_size_lon

        # slice original
        global_patch = arr.sel(time=t_val).isel(
            channel=channel,
            latitude=slice(lat_start, lat_start + H),
            longitude=slice(lon_start, lon_start + W),
        ).values

        tile_patch = tiles.isel(batch_dim=tile_index, channel=channel).values

        # handle padding: tiles fillna(0) -> global_patch may have NaNs at edges, make comparable
        global_patch = np.nan_to_num(global_patch, nan=0.0)

        max_abs = np.max(np.abs(tile_patch - global_patch))
        print(f"Tile {tile_index} vs global slice max abs diff: {max_abs:.6e}")
        assert max_abs <= atol, f"Tile mismatch vs global slice (max_abs={max_abs})"

    #Finally the create dataset method - for the base class, we just return the value tiles without any position info. The subclasses below will override this method to also include positions (either per-pixel or bounds) in the dataset.
    def _create_datasets(self, stage: str | None = None) -> None:
        """
        Create train/val/test datasets.

        Base class here returns values only (no position info):
            each item is a tensor shaped (C, H, W)

        For positional models (inputs_positioned="pixels"/"bounds"), use subclasses below.
        """
        tiles = self._tile_array(self._raw_data)  # (N, C, H, W)

        base = TensorDataset(torch.tensor(tiles.values, dtype=torch.float32))
        train_base, val_base, test_base = self._split_dataset(base)

        self.train_dataset = TransformedInputDataset(train_base, self.train_transforms)
        self.val_dataset = TransformedInputDataset(val_base, self.val_transforms)
        self.test_dataset = (
            None
            if test_base is None
            else TransformedInputDataset(test_base, self.test_transforms)
        )


class ERA5SpatialPixelsDataModule(ERA5TiledBaseDataModule):
    """
    Like the base ERA5 module, but returns per-pixel positions as well: 
        (values, positions)

    positions are a tensor shaped (2, H, W):
        [0, :, :] = latitude
        [1, :, :] = longitude

    This matches what STPatchEmbed expects (values, positions) when we configure mmmae with:
        inputs_positioned="pixels"
    """
    
    # #Extract Real World Coordinates from the tile local coordinates inside the tile.
    def _extract_pixel_positions(self, tiles: xr.DataArray) -> torch.Tensor:
        """
        Build REAL per-pixel (lat, lon) grids for each tile.

        Output shape: (N, 2, H, W)
        - channel 0: latitude degrees
        - channel 1: longitude degrees
        """

        # Get mapping from batch_dim -> (time, tile_lat_id, tile_lon_id)
        # batch_dim coordinate contains tuples because we stacked those dims.
        # Example element: (np.datetime64('2000-01-01T00'), 3, 10)
        
        # print(tiles.shape)
        # print(tiles["batch_dim"].values)
        
        #Grab the mapping for each tile (batch_tuples[0]  = (2005-01-01 00:00, tile_lat_id=0,  tile_lon_id=0)) - which part of the globe this tile came from
        batch_tuples = tiles["batch_dim"].values  # length N

        # print(batch_tuples.shape)
        N = tiles.sizes["batch_dim"]
        H = tiles.sizes["tile_lat"] # == tile_size
        W = tiles.sizes["tile_lon"]

        # print(H, W)
        
        # Global real coordinate arrays (degrees)
        lat_global = self._global_lat  # shape (n_lat,)
        lon_global = self._global_lon  # shape (n_lon,)

        pos = torch.empty((N, 2, H, W), dtype=torch.float32)

        #For each tile, compute the correct slice of real coords based on the tile's position in the global grid (tile_lat_id, tile_lon_id) and the global lat/lon arrays. This way we get real-world lat/lon for each pixel in the tile, which is what the positional embedding expects.
        for i, (_, tile_lat_id, tile_lon_id) in enumerate(batch_tuples):
            lat_start = int(tile_lat_id) * self.tile_size_lat #tile_lat_id is a “block coordinate”, and multiplying by tile size converts it to a lat/lon starting “pixel coordinate”.
            lon_start = int(tile_lon_id) * self.tile_size_lon

            lat_slice = lat_global[lat_start : lat_start + H]
            lon_slice = lon_global[lon_start : lon_start + W]
            
            # if i==551: 
            #     print(f"Tile {i}: tile_lat_id={tile_lat_id}, tile_lon_id={tile_lon_id}")
            #     print("lat_slice:", lat_slice)
            #     print("lon_slice:", lon_slice)
            
            # If boundary="pad" created partial tiles at edges, slices can be shorter. We pad those to H/W using the last available coordinate.
            if len(lat_slice) < H:
                # print(f"Tile {i} has short lat slice of length {len(lat_slice)} (expected {H}), padding to full tile size.")
                # print("Original lat_slice:", lat_slice)
                lat_slice = torch.tensor(lat_slice, dtype=torch.float32)
                lat_slice = torch.cat([lat_slice, lat_slice[-1].repeat(H - len(lat_slice))])
                lat_slice = lat_slice.numpy()
                # print("Padded lat_slice:", lat_slice)
            if len(lon_slice) < W:
                lon_slice = torch.tensor(lon_slice, dtype=torch.float32)
                lon_slice = torch.cat([lon_slice, lon_slice[-1].repeat(W - len(lon_slice))])
                lon_slice = lon_slice.numpy()

            # print(lat_slice)
            # Convert 1D coords to full 2D per-pixel grids - Make (H, W) grids ==  Broadcast the 1D lat/lon slices across the tile dimensions to get full per-pixel coordinate grids for this tile.
            #Lat constant across rows, lon constant across columns, which matches the structure of the global grid and what the model expects for positional encoding.
            lat_2d = torch.tensor(lat_slice, dtype=torch.float32).view(H, 1).repeat(1, W)
            lon_2d = torch.tensor(lon_slice, dtype=torch.float32).view(1, W).repeat(H, 1)

            # print(lat_2d)
            pos[i, 0] = lat_2d
            pos[i, 1] = lon_2d

        return pos
    
    #Prove positions match global coord
    def assert_positions_match_coords(self, tile_index: int = 0, atol: float = 1e-6):
        arr = self._raw_data
        tiles = self._tile_array(arr)
        pos = self._extract_pixel_positions(tiles)  # (N,2,H,W)

        t_val, tile_lat_id, tile_lon_id = tiles["batch_dim"].values[tile_index]
        tile_lat_id = int(tile_lat_id); tile_lon_id = int(tile_lon_id)
        H = tiles.sizes["tile_lat"]; W = tiles.sizes["tile_lon"]
        lat_start = tile_lat_id * self.tile_size_lat
        lon_start = tile_lon_id * self.tile_size_lon

        lat_expected = torch.tensor(self._global_lat[lat_start:lat_start+H], dtype=torch.float32)  # (H,)
        lon_expected = torch.tensor(self._global_lon[lon_start:lon_start+W], dtype=torch.float32)  # (W,)

        # broadcast
        lat_grid_expected = lat_expected.view(H,1).repeat(1,W)
        lon_grid_expected = lon_expected.view(1,W).repeat(H,1)

        lat_grid = pos[tile_index, 0]
        lon_grid = pos[tile_index, 1]

        # handle edge padding if any (your code pads by repeating last coord)
        # so compare only the valid region
        H_valid = min(H, lat_expected.numel()) #numel returns the total number of elements in the tensor, which is just the length for 1D tensors. This handles cases where the tile extends beyond the global grid and was padded, ensuring we only compare the actual valid coordinates.
        W_valid = min(W, lon_expected.numel())
        
        max_abs_lat = torch.max(torch.abs(lat_grid[:H_valid,:W_valid] - lat_grid_expected[:H_valid,:W_valid]))
        max_abs_lon = torch.max(torch.abs(lon_grid[:H_valid,:W_valid] - lon_grid_expected[:H_valid,:W_valid]))
        print(f"Validating tile {tile_index} positions against global coords...")
        print(f"Max abs diff lat: {max_abs_lat}, max abs diff lon: {max_abs_lon}")

        assert max_abs_lat <= atol, f"lat grid mismatch: {max_abs_lat}"
        assert max_abs_lon <= atol, f"lon grid mismatch: {max_abs_lon}"
    
    def _create_datasets(self, stage: str | None = None) -> None:
        """
        Create train/val/test datasets for position models (inputs_positioned="pixels")
        """
        #First create tiles with the local position coords and then compute tile-specific lon/lat grids using the global lon/lat + tile indices and create positions per sample(done by _extract_pixel_positions)
        tiles = self._tile_array(self._raw_data)  # (N, C, H, W)
        positions = self._extract_pixel_positions(tiles)  # (N, 2, H, W)

        print("tiles:", tiles.shape)          # expected (N, C, H, W)
        print("positions:", positions.shape)  # expected (N, 2, H, W)
        base = TensorDataset(
            torch.tensor(tiles.values, dtype=torch.float32),
            positions,
        )
        train_base, val_base, test_base = self._split_dataset(base)

        self.train_dataset = TransformedInputCoordsDataset(
            train_base, self.train_transforms
        )
        self.val_dataset = TransformedInputCoordsDataset(val_base, self.val_transforms)
        self.test_dataset = (
            None
            if test_base is None
            else TransformedInputCoordsDataset(test_base, self.test_transforms)
        )


class ERA5SpatialBoundsDataModule(ERA5TiledBaseDataModule):
    """
    Like the base ERA5 module, but returns *tile bounds* instead of per-pixel coords:
        (values, bounds)

    bounds are shaped (2, 2):
        bounds[0] = [lat_min, lat_max]
        bounds[1] = [lon_min, lon_max]

    This matches what BoundedPatchEmbed expects when you configure mmmae with:
        inputs_positioned="bounds"
    """
    #Same as extract pixel coords but here we extract bounds
    def _extract_bounds(self, tiles: xr.DataArray) -> torch.Tensor:
        """
        Build REAL bounds per tile using the stored global ERA5 lat/lon coordinate axes.

        Output: (N, 2, 2)
        bounds[i, 0] = [lat_min, lat_max]
        bounds[i, 1] = [lon_min, lon_max]
        """
        batch_tuples = tiles["batch_dim"].values  # tuples: (time, tile_lat_id, tile_lon_id)

        N = tiles.sizes["batch_dim"]
        H = tiles.sizes["tile_lat"]
        W = tiles.sizes["tile_lon"]

        lat_global = self._global_lat
        lon_global = self._global_lon

        bounds = torch.empty((N, 2, 2), dtype=torch.float32)

        #Same logic as _extract_pixel_positions, but we just take the min/max lat/lon for the tile's position in the global grid to get the bounds. This way we get real-world lat/lon bounds for each tile, which is what the BoundedPatchEmbed expects for positional encoding.
        for i, (_, tile_lat_id, tile_lon_id) in enumerate(batch_tuples):
            lat_start = int(tile_lat_id) * self.tile_size_lat
            lon_start = int(tile_lon_id) * self.tile_size_lon

            lat_slice = lat_global[lat_start : lat_start + H]
            lon_slice = lon_global[lon_start : lon_start + W]

            # Handle padded edge tiles (same idea as pixel positions)
            if len(lat_slice) == 0 or len(lon_slice) == 0:
                raise RuntimeError("Empty lat/lon slice when computing bounds.")

            lat_min = float(min(lat_slice))
            lat_max = float(max(lat_slice))
            lon_min = float(min(lon_slice))
            lon_max = float(max(lon_slice))

            bounds[i, 0, 0] = lat_min
            bounds[i, 0, 1] = lat_max
            bounds[i, 1, 0] = lon_min
            bounds[i, 1, 1] = lon_max
            
            if i<2:
                print(bounds[i])

        return bounds
        
    def _create_datasets(self, stage: str | None = None) -> None:
        """
        Create train/val/test datasets for position models (inputs_positioned="bounds")
        """
        
        tiles = self._tile_array(self._raw_data)  # (N, C, H, W)
        print("Tiles shape:", tiles.shape)
        bounds = self._extract_bounds(tiles)  # (N, 2, 2)
        
        # print("tiles:", tiles.shape)          # expected (N, C, H, W)
        # print("positions:", positions.shape)  # expected (N, 2, H, W)
        
        base = TensorDataset(
            torch.tensor(tiles.values, dtype=torch.float32),
            bounds,
        )
        train_base, val_base, test_base = self._split_dataset(base)

        self.train_dataset = TransformedInputCoordsDataset(
            train_base, self.train_transforms
        )
        self.val_dataset = TransformedInputCoordsDataset(val_base, self.val_transforms)
        self.test_dataset = (
            None
            if test_base is None
            else TransformedInputCoordsDataset(test_base, self.test_transforms)
        )

def validate_batch(values: torch.Tensor, pos: torch.Tensor, *, lon_mode: str = "[-180,180]"):
    """
    values: (B, C, H, W)
    pos:    (B, 2, H, W)  where pos[:,0]=lat, pos[:,1]=lon
    """

    assert values.ndim == 4, f"values should be (B,C,H,W), got {values.shape}"
    assert pos.ndim == 4, f"pos should be (B,2,H,W), got {pos.shape}"
    B, C, H, W = values.shape
    assert pos.shape == (B, 2, H, W), f"pos shape mismatch, got {pos.shape}"

    assert values.dtype == torch.float32, f"values dtype should be float32, got {values.dtype}"
    assert pos.dtype == torch.float32, f"pos dtype should be float32, got {pos.dtype}"

    # No NaNs/inf unless you explicitly allow them
    assert torch.isfinite(values).all(), "values contain NaN/Inf"
    assert torch.isfinite(pos).all(), "pos contain NaN/Inf"

    lat = pos[:, 0]  # (B,H,W)
    lon = pos[:, 1]  # (B,H,W)

    # Range checks
    assert (lat >= -90.0).all() and (lat <= 90.0).all(), "lat out of range [-90,90]"

    if lon_mode == "[-180,180]":
        assert (lon >= -180.0).all() and (lon <= 180.0).all(), "lon out of range [-180,180]"
    elif lon_mode == "[0,360)":
        assert (lon >= 0.0).all() and (lon < 360.0).all(), "lon out of range [0,360)"
    else:
        raise ValueError(f"Unknown lon_mode={lon_mode}")

    # Structural checks:
    # latitude should be constant across columns within a row
    # i.e. lat[b, r, :] all same
    lat_col_var = lat.var(dim=2)  # variance across W, shape (B,H)
    assert (lat_col_var < 1e-10).all(), "Latitude varies across columns -> wrong broadcasting or transform misalignment"

    # longitude should be constant across rows within a column
    lon_row_var = lon.var(dim=1)  # variance across H, shape (B,W)
    assert (lon_row_var < 1e-10).all(), "Longitude varies across rows -> wrong broadcasting or transform misalignment"

    # Monotonicity checks (within tolerance)
    # lat usually decreases with row index in ERA5 (90 to -90). Either direction is OK but must be monotonic.
    lat_1d = lat[:, :, 0]  # (B,H) pick first column
    dlat = lat_1d[:, 1:] - lat_1d[:, :-1]
    assert ((dlat >= -1e-6).all() or (dlat <= 1e-6).all()), "Latitude not monotonic down rows"

    lon_1d = lon[:, 0, :]  # (B,W) pick first row
    dlon = lon_1d[:, 1:] - lon_1d[:, :-1]
    assert (dlon >= -1e-6).all(), "Longitude not increasing across columns (expected after sortby)"

    # Quick stat sanity
    assert values.abs().mean() > 0, "values look all zeros (did fillna wipe data?)"

#Check to see if lat/lon have real-world magnitude and were not accidentally normalized to 0-1. This is a common mistake that can happen if you forget to disable normalization in your transforms, and it would break the positional encoding since the model expects real lat/lon values.
def assert_positions_not_normalized(pos: torch.Tensor):
    """
    Ensures lat/lon have real-world magnitude and
    were not accidentally normalized to 0-1.
    """
    lat = pos[:, 0]
    lon = pos[:, 1]

    lat_range = lat.max() - lat.min()
    lon_range = lon.max() - lon.min()

    assert lat_range > 10, (
        f"Latitude range too small ({lat_range}). "
        "Positions may have been normalized incorrectly."
    )

    assert lon_range > 10, (
        f"Longitude range too small ({lon_range}). "
        "Positions may have been normalized incorrectly."
    )
    
if __name__ == "__main__":
    
    #For a demo with MMMAE
    from FRAME_FM.models.mmmae import MultimodalMaskedAutoencoder
    
    # 1. Create DataModule - with the module you want
    print("Starting ERA5 dataloader demo...")
    data_module = ERA5SpatialPixelsDataModule(
        data_root="https://gws-access.jasmin.ac.uk/public/eds_ai/era5_repack/aggregations/data/ecmwf-era5X_oper_an_sfc_2000_2020_2d_repack.kr1.0.json",
        variables=["d2m"],  
        time_min="2005-01-01T00",
        time_max="2005-01-01T01",  # tiny subset for quick test
        tile_size_lat=64,
        tile_size_lon=64,
        batch_size=4,
        chunks={"time": 1},  # stream one timestep at a time
        num_workers=0,  # for debugging/demo, set to 0 to avoid multiprocessing issues in notebooks
    )
    
    # data_module = ERA5SpatialBoundsDataModule(  # change to ERA5TiledBaseDataModule if needed
    #     data_root="https://gws-access.jasmin.ac.uk/public/eds_ai/era5_repack/aggregations/data/ecmwf-era5X_oper_an_sfc_2000_2020_2d_repack.kr1.0.json",
    #     variables=["d2m"],  # keep small for demo
    #     time_min="2005-01-01T00",
    #     time_max="2005-01-01T01",  # tiny subset for quick test
    #     tile_size_lat=64,
    #     tile_size_lon=64,
    #     batch_size=4,
    #     chunks={"time": 1},  # stream one timestep at a time
    #     num_workers=0,  # for debugging/demo, set to 0 to avoid multiprocessing issues in notebooks
    # )
        
    print("Setting up DataModule...")
    data_module.setup()
    
    #Working checks
    Debug = True
    if Debug:
        
        # 1) tile/global identity
        for idx in [0, 10, 100]:
            data_module.assert_tile_matches_global(tile_index=idx, channel=0)

        # 2) positions correctness (pixels module only)
        for idx in [0, 10, 100]:
            data_module.assert_positions_match_coords(tile_index=idx)

        # 3) batch invariants (after transforms)
        train_loader = data_module.train_dataloader()
        for _ in range(3):
            values, pos = next(iter(train_loader))
            validate_batch(values, pos, lon_mode="[-180,180]")
            assert_positions_not_normalized(pos)

        print("✅ All dataloader verification checks passed")
            
    print("\nDataset sizes:")
    print("Train:", len(data_module.train_dataset))
    print("Val  :", len(data_module.val_dataset))
    print("Test :", 0 if data_module.test_dataset is None else len(data_module.test_dataset))

    train_loader = data_module.train_dataloader()
    val_loader = data_module.val_dataloader()

    train_batch = next(iter(train_loader))
    val_batch = next(iter(val_loader))

    train_values, train_pos = train_batch
    val_values, val_pos = val_batch

    print("\nTrain batch shapes:")
    print("values:", train_values.shape)
    print("values mean:", train_values.mean().item())
    print("pos   :", train_pos.shape)

    print("\nVal batch shapes:")
    print("  values:", val_values.shape)
    print("  values mean:", val_values.mean().item())
    print("  pos   :", val_pos.shape)
    # print(train_pos)
    # print("\n")
    # print(train_pos[1])
    
    plotting = False  # set to False to skip plotting
    if plotting:
         
        import matplotlib.pyplot as plt
        import numpy as np

        def plot_global_and_tile(arr, tiles, tile_index=0, channel=0, time_index=0):
            """
            arr: xarray DataArray (channel, time, latitude, longitude)
            tiles: xarray DataArray (batch_dim, channel, tile_lat, tile_lon)
            """
            # Global slice
            global_img = arr.isel(channel=channel, time=time_index).values
            plt.figure()
            plt.imshow(global_img, aspect="auto")
            plt.title(f"Global field (channel={channel}, time_index={time_index})")
            plt.colorbar()
            plt.show()

            # Tile slice
            tile_img = tiles.isel(batch_dim=tile_index, channel=channel).values
            plt.figure()
            plt.imshow(tile_img, aspect="auto")
            plt.title(f"Tile (tile_index={tile_index}, channel={channel})")
            plt.colorbar()
            plt.show()
            
        arr = data_module._raw_data           # (channel, time, lat, lon)
        tiles = data_module._tile_array(arr)  # (N, C, H, W) as xarray
        plot_global_and_tile(arr, tiles, tile_index=10, channel=0, time_index=0)
        
        
        def plot_tile_positions(tiles, pos_tensor, tile_index=0):
            """
            tiles: xarray tiles (batch_dim, channel, tile_lat, tile_lon)
            pos_tensor: torch.Tensor (N, 2, H, W) from _extract_pixel_positions
            """
            import torch

            tile_img = tiles.isel(batch_dim=tile_index, channel=0).values
            # print(pos_tensor)
            lat = pos_tensor[tile_index, 0].cpu().numpy()
            lon = pos_tensor[tile_index, 1].cpu().numpy()

            plt.figure()
            plt.imshow(tile_img, aspect="auto")
            plt.title("Tile values (channel 0)")
            plt.colorbar()
            plt.show()

            plt.figure()
            plt.imshow(lat, aspect="auto")
            plt.title("Tile latitude grid")
            plt.colorbar()
            plt.show()

            plt.figure()
            plt.imshow(lon, aspect="auto")
            plt.title("Tile longitude grid")
            plt.colorbar()
            plt.show()
        
        positions = data_module._extract_pixel_positions(tiles)
        plot_tile_positions(tiles, positions, tile_index=10)
        
        def plot_batch_sample(values, pos=None, b=0, c=0):
            """
            values: torch.Tensor (B, C, H, W)
            pos:    torch.Tensor (B, 2, H, W) or None
            """
            img = values[b, c].detach().cpu().numpy()
            plt.figure()
            plt.imshow(img, aspect="auto")
            plt.title(f"Batch sample values (b={b}, c={c})")
            plt.colorbar()
            plt.show()

            if pos is not None:
                lat = pos[b, 0].detach().cpu().numpy()
                lon = pos[b, 1].detach().cpu().numpy()

                plt.figure()
                plt.imshow(lat, aspect="auto")
                plt.title(f"Batch sample lat (b={b})")
                plt.colorbar()
                plt.show()

                plt.figure()
                plt.imshow(lon, aspect="auto")
                plt.title(f"Batch sample lon (b={b})")
                plt.colorbar()
                plt.show()
                
        plot_batch_sample(train_values, train_pos, b=0, c=0)
        
#     #Debug through Python file -- Not working rn
#     from FRAME_FM.dataloaders.era5_dl_verification import (
#     BatchChecksConfig,
#     validate_batch,
#     assert_tile_matches_global,
#     assert_positions_match_coords,
# )
#     print("Setting up DataModule...")
#     data_module = ERA5SpatialPixelsDataModule(
#         data_root="https://gws-access.jasmin.ac.uk/public/eds_ai/era5_repack/aggregations/data/ecmwf-era5X_oper_an_sfc_2000_2020_2d_repack.kr1.0.json",
#         variables=["d2m"],  
#         time_min="2005-01-01T00",
#         time_max="2005-01-01T01",  # tiny subset for quick test
#         tile_size_lat=64,
#         tile_size_lon=64,
#         batch_size=4,
#         chunks={"time": 1},  # stream one timestep at a time
#         num_workers=0,  # for debugging/demo, set to 0 to avoid multiprocessing issues in notebooks
#     )
#     data_module.setup() 
# # after data_module.setup()

#     cfg = BatchChecksConfig(
#         lon_mode="[-180,180]" if data_module.convert_longitude_to_180 else "[0,360)",
#         tile_global_atol=1e-5,
#     )

#     # 1) Check tiling correctness by comparing a few random tiles to raw global slices
#     raw_arr = data_module._raw_data
#     tiles_xr = data_module._tile_array(raw_arr)

#     for idx in [0, 10, 100]:
#         assert_tile_matches_global(
#             raw_arr=raw_arr,
#             tiles=tiles_xr,
#             tile_index=idx,
#             channel=0,
#             tile_size_lat=data_module.tile_size_lat,
#             tile_size_lon=data_module.tile_size_lon,
#             atol=cfg.tile_global_atol,
#         )

#     print("✅ tile->global identity checks passed")

#     # 2) If you are using pixels-positioned datamodule, verify positions match global coords
#     if isinstance(data_module, ERA5SpatialPixelsDataModule):
#         pos_all = data_module._extract_pixel_positions(tiles_xr)

#         for idx in [0, 10, 100]:
#             assert_positions_match_coords(
#                 global_lat=data_module._global_lat,
#                 global_lon=data_module._global_lon,
#                 tiles=tiles_xr,
#                 pos=pos_all,
#                 tile_index=idx,
#                 tile_size_lat=data_module.tile_size_lat,
#                 tile_size_lon=data_module.tile_size_lon,
#                 atol=1e-6,
#             )

#         print("✅ positions match global coord checks passed")

#     # 3) Validate *post-transform* batches coming from the DataLoader (this catches transform misalignment)
#     train_loader = data_module.train_dataloader()

#     for _ in range(3):
#         batch = next(iter(train_loader))
#         if isinstance(data_module, ERA5SpatialPixelsDataModule):
#             values, pos = batch
#             validate_batch(values, pos, cfg=cfg)
#         else:
#             # base module returns values only
#             (values,) = batch
#             validate_batch(values, None, cfg=cfg)

#     print("✅ post-transform batch checks passed")
   
    
    # 2. Fetch one batch
    # train_loader = data_module.train_dataloader()
    # # print("Train loader created:", train_loader.train_dataset)
    # # print(train_loader)
    # batch = next(iter(train_loader))

    # values, positions = batch  # since we used ERA5SpatialPixelsDataModule

    # print("\nBatch loaded successfully.")
    # print("Values shape:", values.shape)      # Expected: (B, C, H, W)
    # print("Positions shape:", positions.shape)  # Expected: (B, 2, H, W)
    
    # print(values)
    # print(positions)

    # B, C, H, W = train_values.shape
    
    # # 3. Create MMMAE model
    # print("\nCreating MMMAE model...")

    # model = MultimodalMaskedAutoencoder(
    #     input_shapes=[(H, W)],
    #     n_channels=[C],
    #     patch_shapes=[(16, 16)],        # patch size inside each tile
    #     inputs_positioned="pixels",     # must match dataloader
    #     position_space=((-90.0, 90.0), (-180.0, 180.0)),
    #     pos_embed_ratio = (1.0, 1.0),     # how much to scale the positional embeddings relative to the value embeddings
    #     encoder_embed_dim=256,
    #     encoder_depth=4,
    #     encoder_num_heads=8,
    # )

    # print("Model created.")

    # # 4. Forward pass
    # print("\nRunning forward pass...")

    # # MMMAE expects list of inputs (one per modality)
    # outputs = model(
    #     inputs=[train_values],
    #     input_positions=[train_pos],
    #     mask_ratio=0.5,
    # )

    # print("Forward pass successful!")

    # # ------------------------------------------------------------------
    # # 5. Inspect outputs
    # # ------------------------------------------------------------------
    # print("\nOutput keys:", outputs.keys())

    # if "reconstructions" in outputs:
    #     recon = outputs["reconstructions"][0]
    #     print("Reconstruction shape:", recon.shape)
    #     # Expected: (B, C, H, W)

    # print("\nDemo completed successfully.")
    
    


Starting ERA5 dataloader demo...
Setting up DataModule...
64 64
tiles: (552, 1, 64, 64)
positions: torch.Size([552, 2, 64, 64])
Tile 0 vs global slice max abs diff: 0.000000e+00
Tile 10 vs global slice max abs diff: 0.000000e+00
Tile 100 vs global slice max abs diff: 0.000000e+00
64 64
Validating tile 0 positions against global coords...
Max abs diff lat: 0.0, max abs diff lon: 0.0
64 64
Validating tile 10 positions against global coords...
Max abs diff lat: 0.0, max abs diff lon: 0.0
64 64
Validating tile 100 positions against global coords...
Max abs diff lat: 0.0, max abs diff lon: 0.0
✅ All dataloader verification checks passed

Dataset sizes:
Train: 469
Val  : 82
Test : 1

Train batch shapes:
values: torch.Size([4, 1, 64, 64])
values mean: 244.56321716308594
pos   : torch.Size([4, 2, 64, 64])

Val batch shapes:
  values: torch.Size([4, 1, 64, 64])
  values mean: 242.68380737304688
  pos   : torch.Size([4, 2, 64, 64])


/home/users/rajat15/FRAME-FM/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
